# Chemprop Tutorial: Predicting Molecular Properties with Chemprop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/instadeepai/alf/blob/main/tutorials/models/chemprop_tutorial.ipynb)

This tutorial demonstrates the `ChempropModel`, an **MPNN (Message Passing Neural Network)** surrogate model backed by [Chemprop v2](https://github.com/chemprop/chemprop).
MPNNs learn directly from molecular graph topology — no hand-crafted fingerprints needed.

> **Prerequisites:** Familiarity with the ALF core API (`Candidate`, `LabelledCandidates`).
> See the [models overview](../extending_base_classes/models.ipynb) for an introduction.

### What you'll learn
1. How to represent molecules as `Candidate` objects with SMILES strings
2. How to configure and train a `ChempropModel` on labelled molecular data
3. How to predict properties for unseen molecules
4. How to inspect per-epoch training metrics
5. How to read final training summary metrics

## Setup

Run the cell below to install ALF and this tutorial's dependencies — **no repository clone required**, so it works in a fresh environment or on Google Colab.

- Already set up a dev environment from a clone (`uv sync`)? You can **skip the install cell**.
- To run on a **GPU**, uncomment the GPU line in the install cell.

For all installation options, see the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

In [ ]:
# Install ALF + this tutorial's dependencies — no clone needed.
# (Skip this cell if you are already running from a cloned repo via `uv sync`.)
%pip install "alf_core @ git+https://github.com/instadeepai/alf.git#subdirectory=core" "alf_tools[chemprop] @ git+https://github.com/instadeepai/alf.git#subdirectory=tools" matplotlib
# GPU (optional): run this AFTER the line above to switch PyTorch to a CUDA build.
# %pip install torch --index-url https://download.pytorch.org/whl/cu128
# Once ALF is on PyPI this simplifies to e.g. `%pip install alf_tools` (no git URL).

We import the Chemprop model wrapper and its configuration classes, plus NumPy and matplotlib for the toy dataset and plots:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from alf_core import Candidate, LabelledCandidates
from alf_tools.models.chemprop import ChempropModel, ChempropModelConfig, ChempropTrainConfig

---
## Section 1: Building a toy molecular dataset

We use 16 small organic molecules with approximate Wildman–Crippen logP values as labels.
Higher logP → more lipophilic (less water-soluble). Real experiments would substitute
experimental assay data here.


In [ ]:
TRAIN_SMILES = [
    "C",  # methane
    "CC",  # ethane
    "CCC",  # propane
    "CCCC",  # butane
    "CCO",  # ethanol
    "CCCO",  # propanol
    "c1ccccc1",  # benzene
    "Cc1ccccc1",  # toluene
    "CC(=O)O",  # acetic acid
    "CC(=O)OCC",  # ethyl acetate
    "CN",  # methylamine
    "CC#N",  # acetonitrile
    "c1ccncc1",  # pyridine
    "c1ccc(cc1)O",  # phenol
    "CC(=O)N",  # acetamide
    "c1ccc(cc1)N",  # aniline
]

# Approximate logP values (Wildman-Crippen)
TRAIN_LABELS = [
    1.1,
    1.8,
    2.4,
    2.9,
    -0.3,
    0.3,
    1.6,
    2.0,
    -0.2,
    0.7,
    -0.6,
    -0.3,
    0.7,
    1.5,
    -1.3,
    0.9,
]

candidates = [Candidate(data=smi, modality="graph") for smi in TRAIN_SMILES]
train_data = LabelledCandidates(candidates, np.array(TRAIN_LABELS))

print(f"Training set: {len(train_data)} molecules")
print(f"Label range:  [{train_data.labels.min():.2f}, {train_data.labels.max():.2f}]")

---
## Section 2: Configuring and training the MPNN

`ChempropModelConfig` controls the graph neural network architecture:
- **`hidden_size`** — dimension of each message-passing hidden layer
- **`depth`** — number of message-passing steps (graph radius)
- **`ffn_num_layers`** — number of feed-forward layers after aggregation
- **`aggregation`** — how node embeddings are pooled to a graph embedding (`mean`/`sum`/`norm`)
- **`dropout`** — dropout rate in the FFN (default 0.0; left at default here)

`ChempropTrainConfig` controls the training loop. We use compact settings here for
fast execution; increase `hidden_size`, `depth`, and `num_epochs` for real datasets.


In [ ]:
model_config = ChempropModelConfig(
    hidden_size=64,  # message-passing hidden dim
    depth=2,  # message-passing steps
    ffn_num_layers=2,  # prediction-head depth
    aggregation="mean",
)

train_config = ChempropTrainConfig(
    learning_rate=1e-3,
    batch_size=8,
    num_epochs=50,
    seed=42,
)

model = ChempropModel(
    name="chemprop_logp_demo",
    model_config=model_config,
    train_config=train_config,
    device="cpu",
)

model.train(train_data)

summary = model.get_training_summary_metrics()
print("Training complete.")
print(f"Final train loss (MSE): {summary['final_train_loss']:.4f}")
print(f"Final train Spearman:   {summary['final_train_spearman']:.3f}")

---
## Section 3: Predicting on unseen molecules

Pass any SMILES as `Candidate` objects with `modality='graph'`. The model handles
featurisation internally — no manual fingerprint computation is needed.

The five molecules below span a range of lipophilicities, giving the model a variety
of graph topologies to process.


In [ ]:
QUERY = {
    "Naphthalene": "c1ccc2ccccc2c1",  # logP ≈  3.3
    "Paracetamol": "CC(=O)Nc1ccc(cc1)O",  # logP ≈  0.5
    "Aspirin": "CC(=O)Oc1ccccc1C(=O)O",  # logP ≈  1.2
    "Caffeine": "Cn1cnc2c1c(=O)n(c(=O)n2C)C",  # logP ≈ -0.1
    "Ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",  # logP ≈  3.97
}

query_candidates = [Candidate(data=smi, modality="graph") for smi in QUERY.values()]
predictions = model.predict(query_candidates)

print(f"{'Molecule':<14}  Predicted logP")
print("-" * 32)
for name, pred in zip(QUERY.keys(), predictions.means):
    print(f"{name:<14}  {pred:+.3f}")

---
## Section 4: Inspecting training metrics

`get_epoch_metrics()` returns a `SurrogateEpochMetrics` object per epoch, containing:
- `train_loss` — mean MSE over the epoch
- `additional_metrics['train_spearman']` — rank correlation on training data
- `additional_metrics['train_mse']` — MSE computed via the `Results` metric registry

A rising Spearman alongside a falling loss confirms the model is learning the correct ranking.


In [ ]:
epoch_metrics = model.get_epoch_metrics()
epochs = [m.epoch for m in epoch_metrics]
train_losses = [m.train_loss for m in epoch_metrics]
train_spearman = [m.additional_metrics.get("train_spearman", float("nan")) for m in epoch_metrics]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_losses, color="#e74c3c", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("MSE Loss")
ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_spearman, color="#3498db", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Spearman ρ")
ax2.set_title("Training Spearman Correlation")
ax2.set_ylim(-1, 1)
ax2.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 5: Training summary metrics

`get_training_summary_metrics()` complements `get_epoch_metrics()` by returning a flat
dictionary of **final** values from the most recent `train()` call — one entry per metric
rather than one object per epoch. Useful for logging, hyperparameter sweeps, or a quick
post-training sanity check without iterating over the epoch list.

Keys follow the pattern `final_{split}_{metric}` and include:
- `final_train_loss` — MSE averaged over the last training epoch
- `final_train_spearman` — Spearman ρ on training data after the last epoch
- `final_train_mse` — MSE from the `Results` metric registry
- `final_val_*` equivalents when `val_data` is provided to `train()`


In [ ]:
summary = model.get_training_summary_metrics()

print(f"{'Metric':<30}  Value")
print("-" * 44)
for key, value in summary.items():
    print(f"{key:<30}  {value:.4f}")

---
## Summary

This tutorial demonstrated the `ChempropModel` MPNN surrogate:

- **Dataset** — represent molecules as `Candidate(data=smiles, modality='graph')`; no fingerprints needed
- **Config** — `ChempropModelConfig` sets architecture (hidden size, depth, aggregation); `ChempropTrainConfig` sets the training loop
- **Train** — `model.train(train_data)` lazy-initialises the network and fits with MSE loss; subsequent calls fine-tune rather than reset
- **Predict** — `model.predict(candidates)` returns `Predictions.means`; no variances (mean-only model)
- **Epoch metrics** — `get_epoch_metrics()` returns one `SurrogateEpochMetrics` per epoch with loss and Spearman ρ
- **Summary metrics** — `get_training_summary_metrics()` returns a flat dict of final train/val losses and correlations

For uncertainty estimates, wrap `ChempropModel` inside `EnsembleWrapper` (see `tutorials/models/ensemble_tutorial.ipynb`).
